In [ ]:
from Models.fast_text.fast_text import FastText
from Datasets.datasets import MyDataset
from sklearn.model_selection import KFold
import copy

#### Loading dataset

In [ ]:
import os
os.environ['NationalityDatasets'] = "./data/"

dt = MyDataset()

In [ ]:
data = dt.get_data()
train = dt.get_train()
test = dt.get_test()
validation = dt.get_validation()
train_validation = dt.get_train_validation()

In [ ]:
def delete_others(dataset, exclude=[]):
    dataset['country'] = dataset.apply(lambda x: 'others' if x.level_3 == 'others' and x.country not in exclude else x.country, axis=1)
    dataset = dataset[dataset.country != 'others']
    return dataset

train = delete_others(train)
test = delete_others(test)
validation = delete_others(validation)
train_validation = delete_others(train_validation)
data = delete_others(data)




In [ ]:
data.shape, train.shape, test.shape, validation.shape, train_validation.shape

In [ ]:
data.level_3.value_counts()

In [ ]:
data.level_3.value_counts().shape

#### Level 3 Classification

In [ ]:
region_3_ft = FastText(data=data, column='level_3')

In [ ]:
region_3_ft.save_train_test(train=train, test=test, validation=validation)

In [ ]:
# %%timeit
# region_3_ft.training()

In [ ]:
# region_3_ft.save_model('level_3_classification.bin')

In [ ]:
region_3_ft.load_model('level_3_classification.bin')

In [ ]:
accuracy = region_3_ft.testing()
print('test accuracy: ', str(accuracy))

In [ ]:
test_df = region_3_ft.load_df()
test_df['pred'] = region_3_ft.predict(test_df['text'])

In [ ]:
region_3_ft.print_classification_metrics(test_df['label'], test_df['pred'])

Micro F1-score 0.6284699543521093


Macro F1-score 0.3690791215952333


Weighted F1-score 0.6145442839376483

Weighted Precision 0.627480001322549


Weighted Recall 0.6284699543521093


Accuracy Score 0.6284699543521093


Balanced Accuracy Score 0.33707752469208224

In [ ]:
region_3_ft.print_classification_report(test_df['label'], test_df['pred'])

In [ ]:
# cm = region_3_ft.confusion_matrix(test_df['label'], test_df['pred'])
# cm1 = cm[:50,:50]
# region_3_ft.display_confusion_matrix(cm1)

#### Analysis

In [ ]:
pred = region_3_ft.decode_label(region_3_ft.predict(test['name']))
pred

In [ ]:
test['label'] = test['level_3']
test['pred'] = pred

In [ ]:
test[(test['pred'] == 'China') & (test['label'] == 'Germany')][['name', 'label', 'pred']].sample(n=10)#.pred.value_counts()#/test[test['label']=='Iran'].shape[0

In [ ]:
test[(test['level_3'] == 'Italy')].pred.value_counts()

### Hierarchical Classification

In [ ]:
region_2_ft = FastText(data=data, column='level_2')
region_2_ft.save_train_test(train=train, test=test, validation=validation)
region_2_labels = region_2_ft.get_labels()
region_2_ft.load_model('level_2_classification.bin')

In [ ]:
fts = []
for i, label in enumerate(region_2_labels):
    print(label)
    
    new_data = data[data.level_2 == label].copy()
    new_train = train[train.level_2 == label].copy()
    new_val = validation[validation.level_2 == label].copy()
    new_test = test[test.level_2 == label].copy()
    
    if len(new_data.level_3.unique()) <= 1:
        continue
    
    new_ft = FastText(type_=label, column='level_3')
    new_ft.save_train_test(train=new_train, test=new_test, validation=new_val)
    
    new_ft.training()
    new_ft.save_model('level_3_' + label+'_classification.bin')
    
    # new_ft.load_model('level_3_' + label + '_classification.bin')
    fts.append(new_ft)
    
    accuracy = new_ft.testing()
    print('test accuracy: ', str(accuracy))
    
    test_df = new_ft.load_df()
    test_df['pred'] = new_ft.predict(test_df['text'])
    
    new_ft.classification_metrics(test_df['label'], test_df['pred'])
    
    # new_ft.classification_report(test_df['label'], test_df['pred'])
    
    cm = new_ft.confusion_matrix(test_df['label'], test_df['pred'])
    new_ft.display_confusion_matrix(cm)

In [ ]:
region_2_ft = FastText(column='level_2')
region_2_ft.save_train_test(train=train, test=test, validation=validation)
region_2_labels = region_2_ft.get_labels()
region_2_ft.load_model('level_2_classification.bin')

In [ ]:
data['predicted_level_2'] = region_2_ft.predict_plus_decode(data['name'])
train['predicted_level_2'] = region_2_ft.predict_plus_decode(train['name'])
test['predicted_level_2'] = region_2_ft.predict_plus_decode(test['name'])
validation['predicted_level_2'] = region_2_ft.predict_plus_decode(validation['name'])

In [ ]:
region_2_labels

In [ ]:
j = 0
new_tests = []
level2_3 = {'Philippines': 'Philippines', 'Chinese': 'China', 'Japanese': 'Japan', 'Korean':'Korea'}
for i, label in enumerate(region_2_labels):
    print(label)
    new_test = test[test.predicted_level_2 == label].copy()
    
    if label in ['Greek', 'Jewish', 'Philippines', 'Chinese', 'Japanese', 'Korean']:
        new_test['predicted_level_3'] = new_test.apply(lambda x: label ,axis=1)
        new_tests.append(new_test)
        continue
    
    # new_ft = FastText(type_=label, column='level_3')
    # new_ft = new_ft.load_model('level_3_' + label + '_classification.bin')
    new_test['predicted_level_3'] = fts[j].predict_plus_decode(new_test['name'])
    new_tests.append(new_test)
    j += 1

In [ ]:
import pandas as pd
t = pd.concat(new_tests)

In [ ]:
region_3_ft.classification_metrics(t['level_3'], t['predicted_level_3'])

In [ ]:
region_3_ft.classification_report(t['level_3'], t['predicted_level_3'])

In [ ]:
cm = region_3_ft.confusion_matrix(t['level_3'], t['predicted_level_3'])
region_3_ft.display_confusion_matrix(cm)

In [ ]:
t['predicted_level_3'].value_counts().index

In [ ]:
t['level_3'].value_counts().index

5-fold cross validation

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)  
accuracies = []
accuracies2 = []
parameters = []
f1_micros = []
f1_macros = []
f1_weighteds = []
recalls = []
precisions = []
balanced_accuracies = []
cms = []
models = []
for train_index, val_index in kfold.split(train_validation):
    
    kfold_train = train_validation.iloc[train_index]
    kfold_validation = train_validation.iloc[val_index]
    print(kfold_train.shape, kfold_validation.shape)
    
    region_3_ft = FastText(data=data, column='level_3')
    region_3_ft.save_train_test(train=kfold_train, test=test, validation=kfold_validation)
    
    region_3_ft.training()
    accuracy = region_3_ft.testing()
    print('test accuracy: ', str(accuracy))
    accuracies.append(accuracy)
    parameters.append(region_3_ft.get_parameters())
    
    
    test_df = region_3_ft.load_df()
    test_df['pred'] = region_3_ft.predict(test_df['text'])
    
    region_3_ft.print_classification_metrics(test_df['label'], test_df['pred'])
    
    accuracies2.append(region_3_ft.accuracy_score(test_df['label'], test_df['pred']))
    f1_micros.append(region_3_ft.f1_micro_score(test_df['label'], test_df['pred']))
    f1_macros.append(region_3_ft.f1_macro_score(test_df['label'], test_df['pred']))
    f1_weighteds.append(region_3_ft.f1_weighted_score(test_df['label'], test_df['pred']))
    recalls.append(region_3_ft.recall_score(test_df['label'], test_df['pred']))
    precisions.append(region_3_ft.precision_score(test_df['label'], test_df['pred']))
    balanced_accuracies.append(region_3_ft.balanced_accuracy_score(test_df['label'], test_df['pred']))
    cms.append(region_3_ft.confusion_matrix(test_df['label'], test_df['pred'], type='precision'))
    
    region_3_ft.save_model(f'level_3_classification_{len(accuracies)}.bin')

In [ ]:
import numpy as np
print('accuracies', np.mean(accuracies))
print('accuracies2', np.mean(accuracies2))
print('f1_micros', np.mean(f1_micros))
print('f1_macros', np.mean(f1_macros))
print('f1_weighteds', np.mean(f1_weighteds))
print('recalls', np.mean(recalls))
print('precisions', np.mean(precisions))
print('balanced_accuracies', np.mean(balanced_accuracies))

In [ ]:
accuracies2